# BÀI TẬP PHẦN 3: BÀI TOÁN N-QUEENS

**Họ tên:** Nguyễn Ngọc Hải

**MSSV:** 24880218

## Hướng dẫn
Trong phần này, bạn sẽ áp dụng các chiến lược tìm kiếm để giải bài toán N-Queens (xếp N quân hậu lên bàn cờ $N \times N$ sao cho không có 2 quân hậu nào ăn nhau).

**Yêu cầu Output:**
- Tất cả các hàm phải trả về **một list các tuple** đại diện cho vị trí của N quân hậu, hoặc `None` nếu không tìm thấy lời giải.
- Định dạng tuple: `(row, col)`.
- Ví dụ với N=4: `[(0, 1), (1, 3), (2, 0), (3, 2)]` (hoặc thứ tự bất kỳ miễn là đúng vị trí).

**Yêu cầu thuật toán:**
1. **DFS (Thay thế Backtracking cổ điển)**: Dùng Stack.
2. **BFS**: Dùng Queue.
3. **UCS**: Dùng Priority Queue với chi phí $g(n)$ là số lượng quân hậu đã đặt.
4. **Greedy**: Dùng Priority Queue với heuristic $h(n)$ (số cặp xung đột). **KHÔNG** kiểm tra tính an toàn (`is_safe`) khi thêm quân để heuristic phát huy tác dụng.
5. **A***: Dùng Priority Queue với $f(n) = g(n) + h(n)$. **KHÔNG** kiểm tra tính an toàn (`is_safe`) khi thêm quân.

**Lưu ý:**
- Các cell kiểm tra (Judge Cells) sẽ sử dụng một hàm kiểm tra tính hợp lệ (check validity) duy nhất để chấm điểm kết quả của bạn, không so sánh cứng với đáp án mẫu.

In [1]:
import time
import heapq
from collections import deque
import matplotlib.pyplot as plt 
%matplotlib inline

### 3.1. Mô hình hóa bài toán

Mô tả bài toán bằng ngôn ngữ toán học (Trạng thái, Phép chuyển, Heuristic, Đích).

Câu trả lời:

YOUR ANSWER HERE

**Yêu cầu bài toán**: Tìm cách đặt N quân hậu trên bàn cờ có N x N ô

**1. Trạng thái (state)**

Một trạng thái là một bộ (tuple) gồm vị trí hàng của các quân hậu đã đặt, với index là chỉ số cột (column) và giá trị tương ứng là chỉ số hàng (row):

$$s = (r_0, r_1, \dots, r_{k-1}), \quad 0 \le k \le N, \quad 0 \le r_i \le (N - 1)$$

- $r_i$ là **hàng** của quân hậu ở **cột** $i$.
- $k = |s|$ là số quân hậu đã đặt; cột tiếp theo cần đặt là cột $k$.

**Trạng thái đầu:** $$s_0 = () \quad \text{(bàn cờ trống, chưa đặt quân nào)}$$

**2. Phép chuyển**

Từ trạng thái $s = (r_0, \dots, r_{k-1})$ với $k < N$, hành động là chọn một hàng $r$ để đặt quân hậu vào cột $k$.

Khi $|s| = N$ không cho phép thực hiện phép chuyển nào (không đặt quá N cột).

**Chi phí** $g(s)$:

Mỗi lần đặt một quân hậu có chi phí bằng 1.

Vậy nên tổng chi phí của một state *(Số lượng quân hậu đã đặt)* cũng chính là kích thước của state.

**3. Heuristic**

$h(s)$ là **số cặp quân hậu tấn công nhau** trong trạng thái $s$:

Điều kiện để cặp quân hậu tấn công nhau:
1. Hai hậu cùng hàng: $r_i = r_j$.
2. Hai hậu cùng đường chéo: $|r_i - r_j| = |j - i|$. Để kiểm tra 2 hậu cùng đường chéo thì hiệu của chỉ số cột và hiệu của chỉ số dòng phải bằng nhau. (*Xét giá trị tuyệt đối*)
2. Không cần xét cùng cột vì mỗi cột trong state chỉ có 1 hậu nên không có trường hợp 2 hậu cùng cột.

Tính chất của heuristic:

- $h(s) = 0$ khi và chỉ khi không có cặp hậu nào tấn công nhau. Đây cũng là giá trị heuristic của trạng thái **"An toàn"**
- Heuristic sẽ **không giảm** mỗi lần thêm 1 hậu mới vào state: việc thêm quân hậu chỉ có thể tạo thêm xung đột, không xóa được xung đột cũ.

**Lưu ý:** Cặp xung đột của 2 quân hậu chỉ tính là 1 cặp xung đột. Ví dụ: 2 hậu xung đột có cùng đường chéo (0, 1) và (1, 0) sẽ tính là 1 cặp xung đột (không tính thành 2 cặp).

**4. Đích**

$$\text{Goal}(s) \iff |s| = N \ \land\ h(s) = 0$$

Tức là:
1. Đã đặt đủ N quân hậu
2. Không có cặp xung đột nào trong state.

Khi state có đủ cả 2 điều kiện trên, state trở thành lời giải của bài toán.

Lời giải được trả về dưới dạng danh sách tọa độ:

$$\text{Output}(s) = [(r_0, 0), (r_1, 1), \dots, (r_{N-1}, N-1)]$$

**5. Hàm đánh giá của từng thuật toán**

| Thuật toán | Cấu trúc dữ liệu | Thứ tự mở rộng | Cắt nhánh (`Pruning`) |
|---|---|---|---|
| BFS | Queue (FIFO) | Theo tầng | Có |
| DFS | Stack (LIFO) | Đi sâu trước | Có |
| UCS | Priority Queue | $f(s) = g(s)$ | Có |
| Greedy | Priority Queue | $f(s) = h(s)$ | Không |
| A\* | Priority Queue | $f(s) = g(s) + h(s)$ | Không |

### 3.2. Các hàm bổ trợ

Cài đặt hàm `count_conflicts` (dùng cho Heuristic) và `is_safe` (dùng cho Pruning các thuật toán mù).

In [2]:
def count_conflicts(state):
    """
    Đếm tổng số cặp hậu tấn công nhau trong trạng thái `state`.
    Input: list/tuple vị trí hàng. VD: (0, 2) nghĩa là cột 0 hàng 0, cột 1 hàng 2.
    Output: integer (số lượng cặp xung đột)
    """
    # YOUR CODE HERE
    count = 0
    # Duyệt ma trận theo cột từ trái sang phải
    for base_col, base_row in enumerate(state):
        # Lấy các cột còn lại so với cột đang duyệt
        for col, row in enumerate(state[base_col + 1:], start=base_col + 1):
            # Count nếu chung hàng hoặc đường chéo
            if row == base_row or col - base_col == abs(row - base_row):
                count += 1

    return count

def is_safe(state, row, col):
    """
    Kiểm tra xem đặt hậu vào (row, col) có an toàn so với các quân hậu trong `state` không.
    state: list/tuple vị trí hàng của các cột trước đó.
    """
    # YOUR CODE HERE
    # Duyệt qua state
    for c, r in enumerate(state):
        # Không an toàn nếu thêm vào cùng hàng hoặc đường chéo
        if row == r or abs(row - r) == abs(col - c):
            return False 

    # An toàn
    return True


### 3.3. Cài đặt các thuật toán

Hãy cài đặt đầy đủ 5 thuật toán bên dưới. Mỗi hàm nhận vào `N` và trả về `List[Tuple[int, int]]` hoặc `None`.

In [3]:
def queens_bfs(N):
    """BFS: Queue, có pruning (dùng is_safe)."""
    # YOUR CODE HERE
    queue = deque([()])
    # Loop đến khi Queue empty
    while queue:
        state = queue.popleft()  # Lấy state cần duyệt
        col = len(state)         # Lấy vị trí cột tiếp theo cần đặt hậu

        # Dừng nếu đã đặt đủ hậu
        if col == N:             
            return [(row, column) for column, row in enumerate(state)]
        
        # Cập nhật các state và queue
        for row in range(N):
            # Add vào queue nếu position an toàn
            if is_safe(state, row, col):
                queue.append((*state, row))  # Tạo state mới với row an toàn

    # Không tìm được kết quả -> Trả về None
    return None

In [4]:
# TEST CELL: BFS (10 points)
print(queens_bfs(9))

[(0, 0), (2, 1), (5, 2), (7, 3), (1, 4), (3, 5), (8, 6), (6, 7), (4, 8)]


In [5]:
def queens_dfs(N):
    """DFS: Stack, có pruning (dùng is_safe)."""
    # YOUR CODE HERE
    stack = [()]

    # Loop đến khi Stack empty
    while stack:
        # Lấy 1 state ra khỏi stack
        state = stack.pop()
        # Tìm index cột cần đặt hậu
        col = len(state)
        # Dừng nếu đã đặt đủ số quân hậu
        if col == N:
            return [(r, c) for c, r in enumerate(state)]

        # Duyệt qua các row trong cột
        for row in reversed(range(N)):
            # Kiểm tra nếu (row, col) an toàn để add vào state
            if is_safe(state, row, col):
                # Cập nhật state và add vào stack
                stack.append((*state, row))
    
    # Không có kết quả -> Trả về None
    return None

In [6]:
# TEST CELL: DFS (10 points)
print(queens_dfs(9))

[(0, 0), (2, 1), (5, 2), (7, 3), (1, 4), (3, 5), (8, 6), (6, 7), (4, 8)]


In [7]:
def queens_ucs(N):
    """UCS: PQ, g=depth, có pruning."""
    # YOUR CODE HERE
    pq = [(0, ())]  # Priority Queue

    # Loop đến khi Priority Queue empty
    while pq:
        # Lấy g và state có trọng số thấp nhất
        g, state = heapq.heappop(pq)
        # Dừng nếu đã đặt đủ N quân hậu
        if g == N:
            return [(r, c) for c, r in enumerate(state)]

        # Tìm index cột cần đặt hậu
        col = len(state)
        # Duyệt các row trong cột
        for row in range(N):
            # Cập nhật state, g và add vào Priority Queue
            if is_safe(state, row, g):
                heapq.heappush(pq, (g + 1, (*state, row)))

    # Không có kết quả -> Trả về None
    return None

In [8]:
# TEST CELL: UCS (10 points)
print(queens_ucs(9))

[(0, 0), (2, 1), (5, 2), (7, 3), (1, 4), (3, 5), (8, 6), (6, 7), (4, 8)]


In [9]:
def queens_greedy(N):
    """Greedy: PQ, h=conflicts, KHÔNG pruning."""
    # YOUR CODE HERE
    pq = [(0, ())]  # Priority Queue

    # Loop đến khi Priority Queue empty
    while pq:
        heuristic, state = heapq.heappop(pq)  # Lấy heuristic thấp nhất và state tương ứng khỏi Priority Queue
        # Dừng nếu đã đặt đủ quân hậu và heuristic == 0
        if len(state) == N:
            if heuristic == 0:
                return [(r, c) for c, r in enumerate(state)]
            continue  # Continue để bỏ qua việc mở rộng thêm state quá size N
        # Thêm hậu ở các row vào state
        for row in range(N):
            new_state = (*state, row)  # Tạo state mới bằng việc thêm 1 quân hậu ở row
            heapq.heappush(pq, (count_conflicts(new_state), new_state))  # Enqueue heuristic và state mới vào Priority Queue

    # Không tìm được lời giải -> Trả về None
    return None

In [10]:
# TEST CELL: Greedy (10 points)
print(queens_greedy(9))

[(0, 0), (2, 1), (5, 2), (7, 3), (1, 4), (3, 5), (8, 6), (6, 7), (4, 8)]


In [11]:
def queens_astar(N):
    """A*: PQ, f=g+h, KHÔNG pruning."""
    # YOUR CODE HERE
    pq = [(0, ())]  # Priority Queue

    # Loop đến khi Priority Queue empty
    while pq:
        _, state = heapq.heappop(pq)  # Lấy state có f thấp nhất ra khỏi Priority Queue
        # Dừng nếu đã đặt đủ quân hậu và không có conflict
        if len(state) == N:
            if count_conflicts(state) == 0:
                return [(r, c) for c, r in enumerate(state)]
            continue  # Bỏ qua các state đủ quân hậu nhưng có conflict
        # Add các row mới vào state
        for row in range(N):
            new_state = (*state, row)  # Tạo state mới bằng việc thêm 1 quân hậu ở row
            f = len(new_state) + count_conflicts(new_state)  # Tính f = g + h
            heapq.heappush(pq, (f, new_state))  # Enqueue với f và state mới vào Priority Queue

    # Không tìm được lời giải -> Trả về None
    return None

In [12]:
# TEST CELL: A* (10 points)
print(queens_astar(9))

[(0, 0), (2, 1), (5, 2), (7, 3), (1, 4), (3, 5), (8, 6), (6, 7), (4, 8)]


### 3.4. So sánh hiệu năng

Chạy các thuật toán với N từ 4 đến 7. So sánh thời gian thực thi.

In [13]:
def benchmark_n_queens():
    ns = range(4, 8)
    algos = [
        ("DFS", queens_dfs),
        ("BFS", queens_bfs),
        ("UCS", queens_ucs),
        ("Greedy", queens_greedy),
        ("A*", queens_astar)
    ]
    
    print(f"{'N':<5} | {'Algo':<10} | {'Time (s)':<10} | {'Result'}")
    print("-"*45)
    
    for n in ns:
        for name, func in algos:
            try:
                start = time.time()
                res = func(n)
                dur = time.time() - start
                found = 'Yes' if res else 'No'
                print(f"{n:<5} | {name:<10} | {dur:.6f}   | {found}")
            except NotImplementedError:
                print(f"{n:<5} | {name:<10} | Not Impl     | -")

benchmark_n_queens()

N     | Algo       | Time (s)   | Result
---------------------------------------------
4     | DFS        | 0.000019   | Yes
4     | BFS        | 0.000035   | Yes
4     | UCS        | 0.000038   | Yes
4     | Greedy     | 0.000076   | Yes
4     | A*         | 0.000180   | Yes
5     | DFS        | 0.000010   | Yes
5     | BFS        | 0.000064   | Yes
5     | UCS        | 0.000131   | Yes
5     | Greedy     | 0.000033   | Yes
5     | A*         | 0.001158   | Yes
6     | DFS        | 0.000059   | Yes
6     | BFS        | 0.000251   | Yes
6     | UCS        | 0.000286   | Yes
6     | Greedy     | 0.000308   | Yes
6     | A*         | 0.010679   | Yes
7     | DFS        | 0.000045   | Yes
7     | BFS        | 0.001480   | Yes
7     | UCS        | 0.001940   | Yes
7     | Greedy     | 0.000207   | Yes
7     | A*         | 0.059218   | Yes


### Câu hỏi thảo luận
Tại sao bài toán N-Queens trở nên rất khó (tốn nhiều thời gian) khi N tăng lên? Độ phức tạp lý thuyết là bao nhiêu?

Câu trả lời:

YOUR ANSWER HERE
#### 1. Không gian trạng thái tăng theo hàm mũ / giai thừa
Chỉ cần tăng N thêm 1, không gian $N^N$ tăng khoảng $e \cdot N$ lần. Thuật toán tìm kiếm mù phải duyệt qua rất nhiều trạng thái "vô ích" trước khi gặp lời giải. $\iff$ N càng lớn thì không gian N x N sẽ càng lớn.

Xét theo từng thuật toán:
- **BFS** duyệt theo tầng nên khi N tăng -> kích thước của 1 tầng tăng lên -> Mất nhiều thời gian để duyệt hết 1 tầng trước khi qua tầng tiếp theo. Lặp lại đến khi đủ N tầng.
- **DFS** duyệt theo chiều sâu nên khi N tăng -> chiều sâu tăng -> Thời gian đào sâu đến tầng N tăng lên, trong trường hợp xấu nhất (chọn sai nhánh) có thể phải duyệt qua hết các nhánh sai mới tìm được kết quả.
- **UCS** với bài toán hiện tại, g chỉ tăng 1 mỗi khi đặt 1 quân hậu nên kết quả duyệt của **UCS** sẽ giống với duyệt bằng **BFS** và không tạo nhiều sự khác biệt.
- **Greedy** nhanh hơn vì mọi trạng thái an toàn đều có $h = 0$; Priority Queue luôn mở rộng trạng thái an toàn trước. Các trạng thái có xung đột được sinh ra nhưng nằm lại trong Priority Queue sẽ làm tốn bộ nhớ khi N tăng (vì số lượng state trong Priority Queue tăng).
- **A\*** chậm nhất và chậm hơn mỗi khi N tăng. Lý do là **A\*** phụ thuộc vào $g$ và $h$. $g$ tăng 1 với mỗi quân được đặt, trong khi $h$ ở các tầng nông còn nhỏ. Nghĩa là: một state sau khi đặt thêm hậu thì sẽ có $f$ tăng lên  ít nhất là 1, nhưng các state chưa được đặt thêm hậu trong Priority Queue thì có thể $f$ vẫn sẽ nhỏ hơn state vừa được cập nhật. Và A\* vì vậy ưu tiên các state có $f$ nhỏ và phải duyệt qua hết các state này trước khi state (tốt nhất) được duyệt tiếp. => Nên khi N tăng lên, thì số state cũng sẽ tăng lên theo N.

#### 2. Độ phức tạp lý thuyết

##### Thời gian

| Thuật toán | Số nút tối đa | Chi phí mỗi nút | Độ phức tạp thời gian |
|---|---|---|---|
| BFS / UCS / DFS **có pruning** (`is_safe`) | $O(N!)$ | $N$ lần gọi `is_safe`, mỗi lần $O(N)$ | $O(N^2 \cdot N!)$ |
| Greedy / A* **không pruning** (trường hợp xấu nhất) | $O(N^N)$ | `count_conflicts` $O(N^2)$ | $O(N^2 \cdot N^N)$ |

- Với pruning: ở cột thứ k chỉ còn tối đa $N - k$ hàng chưa bị chiếm, nên số nút bị chặn trên bởi $N!$. Thực tế còn nhỏ hơn nhiều vì ràng buộc đường chéo loại thêm nhánh.
- Không pruning: cây có $\sum_{k=0}^{N} N^k = O(N^N)$ nút, vì trạng thái có xung đột vẫn được sinh ra và đưa vào hàng đợi.

##### Bộ nhớ

| Thuật toán | Bộ nhớ |
|---|---|
| DFS | $O(N^2)$ – stack giữ tối đa N con cho mỗi tầng trên một nhánh |
| BFS / UCS | $O(b^d)$ – phải lưu toàn bộ tầng sâu nhất (với pruning: cỡ $O(N!)$) |
| Greedy / A* | $O(N^N)$ trong trường hợp xấu nhất – Priority Queue chứa cả trạng thái có xung đột |

Tóm lại, với tìm kiếm có hệ thống trên cây trạng thái, độ phức tạp là **hàm mũ**: $O(N^N)$ nếu không cắt nhánh và $O(N!)$ nếu có cắt nhánh.
